# 01 · Exploração dos dados sintéticos

**Gazeta Meridiano** é uma empresa fictícia; os dados são 100% sintéticos (seed fixa). Este notebook explora o conjunto antes da análise formal: atributos, entradas, censura à direita e as primeiras curvas. A análise A–H está no notebook 02.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
raiz = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(raiz / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from sobrevivencia import config as C, simulate as S, km as K, cohorts as H, cox as X
from sobrevivencia import value as V, pitfalls as P, competing as G, figures as F

pd.options.display.float_format = "{:,.3f}".format
pd.options.display.max_columns = 30
F._estilo()
df, lat = S.simular()
gt = S.construir_ground_truth(df, lat)
import tempfile
FIG = Path(tempfile.mkdtemp(prefix='figuras_nb_'))
print(f"{len(df):,} assinantes | seed {C.SEED} | corte {C.CORTE}")

18,000 assinantes | seed 42 | corte 2026-10-31


## Os dados

Uma linha por assinante. `duracao_meses` é o tempo observado e `evento` vale 1 se cancelou e 0 se foi censurado no corte.

In [2]:
df.head(8)

,id_assinante,data_inicio,canal,plano,oferta_entrada,dispositivo,engajamento_pre,safra,seguimento_max_meses,duracao_meses,evento,causa,data_cancelamento
0,A000001,2023-11-01,busca_paga,mensal,nenhuma,desktop,53.200,S1,36.008,0.553,1,involuntario,2023-11-17
1,A000002,2023-11-01,organico,combo,nenhuma,tablet,40.900,S1,36.008,36.008,0,,NaT
2,A000003,2023-11-01,midia_paga,mensal,nenhuma,desktop,40.800,S1,36.008,14.035,1,voluntario,2025-01-01
3,A000004,2023-11-01,organico,mensal,nenhuma,mobile,42.900,S1,36.008,6.222,1,voluntario,2024-05-08
4,A000005,2023-11-01,email_crm,combo,nenhuma,mobile,33.500,S1,36.008,36.008,0,,NaT
5,A000006,2023-11-01,busca_paga,mensal,nenhuma,mobile,44.500,S1,36.008,25.874,1,voluntario,2025-12-27
6,A000007,2023-11-01,indicacao,combo,nenhuma,mobile,72.200,S1,36.008,36.008,0,,NaT
7,A000008,2023-11-01,organico,anual,nenhuma,tablet,42.300,S1,36.008,24.359,1,fim_contrato,2025-11-11


In [3]:
df.describe(include='all').T[['count', 'unique', 'top', 'mean', 'min', 'max']]

,count,unique,top,mean,min,max
id_assinante,18000,18000,A000001,NaN,NaN,NaN
data_inicio,18000,NaN,NaN,2025-06-01 18:54:04.800000,2023-11-01 00:00:00,2026-10-31 00:00:00
canal,18000,6,organico,NaN,NaN,NaN
plano,18000,4,mensal,NaN,NaN,NaN
oferta_entrada,18000,3,nenhuma,NaN,NaN,NaN
dispositivo,18000,3,mobile,NaN,NaN,NaN
engajamento_pre,"18,000.000",NaN,NaN,51.285,0.000,100.000
safra,18000,12,S9,NaN,NaN,NaN
seguimento_max_meses,"18,000.000",NaN,NaN,16.993,0.033,36.008
duracao_meses,"18,000.000",NaN,NaN,11.690,0.004,36.008


## Entradas e campanhas

A intensidade de entrada tem picos nas campanhas (Black Friday de cada ano, Passe Copa e Eleições 2026).

In [4]:
entradas = df.groupby(df['data_inicio'].dt.to_period('M')).size()
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(entradas.index.astype(str), entradas.values, color=C.COR['meridiano'])
ax.set_title('Novos assinantes por mês de entrada'); ax.tick_params(axis='x', rotation=70)
plt.show()
entradas.describe()

count      36.000
mean      500.000
std       151.156
min       374.000
25%       408.750
50%       439.000
75%       524.750
max     1,103.000
dtype: float64

In [5]:
mix = pd.crosstab(df['safra'], df['canal'], normalize='index')
mix.style.format('{:.1%}').background_gradient(cmap='Blues', axis=None)

canal,busca_paga,email_crm,indicacao,midia_paga,organico,social
safra,,,,,,
S1,18.2%,13.5%,6.9%,24.0%,26.2%,11.2%
S10,18.2%,11.9%,8.5%,19.6%,30.7%,11.2%
S11,16.5%,11.3%,7.7%,22.2%,29.5%,12.8%
S12,15.9%,10.7%,7.1%,16.4%,35.0%,14.8%
S2,17.9%,12.4%,9.1%,19.3%,28.5%,12.8%
S3,20.3%,12.6%,7.3%,18.4%,29.1%,12.3%
S4,17.9%,11.1%,7.2%,21.4%,29.9%,12.6%
S5,18.3%,13.8%,7.3%,25.2%,24.6%,10.8%
S6,17.1%,12.1%,7.9%,20.0%,31.9%,10.9%


In [6]:
pd.concat({c: df[c].value_counts(normalize=True) for c in ['plano', 'oferta_entrada', 'dispositivo']}, axis=1)

,plano,oferta_entrada,dispositivo
mensal,0.457,NaN,NaN
combo,0.200,NaN,NaN
estudante,0.179,NaN,NaN
anual,0.163,NaN,NaN
nenhuma,NaN,0.887,NaN
black_friday,NaN,0.089,NaN
passe_copa,NaN,0.024,NaN
mobile,NaN,NaN,0.614
desktop,NaN,NaN,0.306
tablet,NaN,NaN,0.080


## Censura à direita

O corte é fixo, então quem entrou tarde tem pouco tempo de observação. A figura mostra a vida de 40 assinantes: `x` = cancelou, círculo vazado = censurado (ainda ativo no corte).

In [7]:
amostra = df.sample(40, random_state=C.SEED).sort_values('data_inicio')
fig, ax = plt.subplots(figsize=(10, 5))
for i, (_, r) in enumerate(amostra.iterrows()):
    ax.hlines(i, 0, r['duracao_meses'], color=C.COR['meridiano'], lw=1.4)
    ax.plot(r['duracao_meses'], i, 'x' if r['evento'] else 'o', color=C.COR['vermelho'] if r['evento'] else C.COR['cinza'],
            mfc='none' if not r['evento'] else None)
ax.set_xlabel('meses desde a primeira cobrança'); ax.set_yticks([]); ax.set_title('Linhas de vida (x = cancelou, o = censurado)')
plt.show()

In [8]:
cens = df.groupby('safra').agg(n=('evento', 'size'), cancelaram=('evento', 'sum'), seguimento_min=('seguimento_max_meses', 'min'),
                               seguimento_max=('seguimento_max_meses', 'max'))
cens['pct_censurado'] = 1 - cens['cancelaram'] / cens['n']
cens

,n,cancelaram,seguimento_min,seguimento_max,pct_censurado
safra,,,,,
S1,1568,951,33.018,36.008,0.393
S10,1332,325,6.078,8.969,0.756
S11,1811,394,3.055,6.045,0.782
S12,1875,98,0.033,3.023,0.948
S2,1130,642,30.062,32.986,0.432
S3,1239,740,27.039,30.029,0.403
S4,1276,737,24.016,27.006,0.422
S5,1800,1020,20.994,23.984,0.433
S6,1243,631,18.070,20.961,0.492


## Primeiras curvas

Kaplan-Meier de toda a base e por plano. O plano anual quase não cai até o mês 12 e depois despenca na renovação.

In [9]:
fig, ax = plt.subplots(figsize=(9, 4.6))
F._curva_km(ax, df, C.COR['tinta'], 'Todos')
for p in C.PLANOS:
    F._curva_km(ax, df[df['plano'] == p], C.COR_PLANO[p], C.PLANOS_ROTULO[p], ic=False)
ax.set_xlim(0, 24); ax.set_ylim(0, 1.02); ax.legend(); ax.set_xlabel('meses'); ax.set_title('Kaplan-Meier: base inteira e por plano')
plt.show()

## Risco empírico por mês de vida

O risco mensal (eventos ÷ tempo sob risco) mostra o **penhasco** da promoção: quem entrou pela Black Friday cancela muito mais no mês 3–4, o primeiro a preço cheio.

In [10]:
na = df[df['plano'] != 'anual']
bf, ref = na[na['oferta_entrada'] == 'black_friday'], na[na['oferta_entrada'] == 'nenhuma']
tab = pd.DataFrame({'sem_promocao': [r['taxa'] for r in K.hazard_por_idade(ref['duracao_meses'], ref['evento'], 12)],
                    'black_friday': [r['taxa'] for r in K.hazard_por_idade(bf['duracao_meses'], bf['evento'], 12)]})
tab.index.name = 'mês de vida'
ax = tab.plot(drawstyle='steps-post', figsize=(9, 3.8), color=[C.COR['meridiano'], C.COR['vermelho']])
ax.set_ylabel('risco mensal'); ax.set_title('Risco mensal de cancelar por mês de vida'); plt.show()
tab.round(4)

,sem_promocao,black_friday
mês de vida,,
0,0.052,0.047
1,0.049,0.048
2,0.040,0.042
3,0.039,0.111
4,0.037,0.050
5,0.035,0.049
6,0.040,0.041
7,0.036,0.041
8,0.037,0.046


## O que o KM recupera da verdade plantada

Como o gerador é conhecido, dá para comparar o KM por canal com a sobrevivência verdadeira.

In [11]:
linhas = []
for c in C.CANAIS:
    sub = df[df['canal'] == c]
    for h in (12, 18):
        p = K.km_ponto(sub['duracao_meses'], sub['evento'], h)
        v = gt['sobrevivencia_verdade']['por_canal'][c][f'm{h}']['todos']
        linhas.append({'canal': C.CANAIS_ROTULO[c], 'h': f'M{h}', 'KM': p['valor'], 'IC_lo': p['lo'], 'IC_hi': p['hi'], 'verdade': v,
                       'cobre': p['lo'] <= v <= p['hi']})
pd.DataFrame(linhas)

,canal,h,KM,IC_lo,IC_hi,verdade,cobre
0,Orgânico,M12,0.706,0.692,0.720,0.699,True
1,Orgânico,M18,0.564,0.548,0.581,0.560,True
2,Busca paga,M12,0.650,0.632,0.668,0.647,True
3,Busca paga,M18,0.504,0.483,0.525,0.492,True
4,Mídia paga (Meta),M12,0.550,0.532,0.567,0.537,True
5,Mídia paga (Meta),M18,0.378,0.360,0.397,0.361,True
6,Social,M12,0.596,0.572,0.619,0.607,True
7,Social,M18,0.445,0.419,0.471,0.454,True
8,Indicação,M12,0.792,0.766,0.815,0.805,True
9,Indicação,M18,0.674,0.643,0.703,0.688,True


## Próximo passo

O notebook **02 · Análise** aplica o desenho A–H: ingênuo × KM por safra, log-rank, Cox com a violação mostrada e corrigida, valor por canal, armadilhas e riscos competitivos.